# Reuters Leakage Ablation Experiment

Tests whether the 100% Kaggle DistilBERT accuracy is caused by the "Reuters" token leaking through preprocessing.

**Setup:** trains DistilBERT twice, identically, on the same 15,000-row sample:
1. **WITH Reuters** (`transformer_text` column) — replicates the original 100% result
2. **WITHOUT Reuters** (`transformer_text_no_reuters` column) — Reuters mentions fully stripped (verified: 17,794 articles had it, now 0)

If accuracy drops meaningfully in run 2, that's direct causal evidence (not just correlation) that Reuters-token leakage was driving the inflated benchmark.

**Before running:** Runtime → Change runtime type → T4 GPU → Save. Upload the new `kaggle_clean.csv` (regenerated — has the `transformer_text_no_reuters` column).

In [ ]:
!pip install -q transformers datasets torch scikit-learn pandas

## Imports and setup

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

print("GPU available:", torch.cuda.is_available())

MODEL_NAME = "distilbert-base-uncased"
LABEL_MAP = {"FAKE": 0, "REAL": 1}
MAX_LENGTH = 256

## Reusable training function (takes the text column name as a parameter)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, pos_label=0),
        "recall": recall_score(labels, preds, pos_label=0),
        "f1": f1_score(labels, preds, pos_label=0),
    }


def train_and_eval(text_column, run_name, n_samples=15000, epochs=3, batch_size=16):
    print(f"\n{'='*60}\nRun: {run_name}  (text column: {text_column})\n{'='*60}")

    df = pd.read_csv("kaggle_clean.csv")
    df = df.dropna(subset=[text_column, "binary_label"])
    df["label"] = df["binary_label"].map(LABEL_MAP)
    df = df[[text_column, "label"]].rename(columns={text_column: "text"})
    df = df.sample(n=n_samples, random_state=42).reset_index(drop=True)

    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
    print(f"Train: {len(train_df)} | Test: {len(test_df)}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    def tokenize(batch):
        return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

    train_ds = Dataset.from_pandas(train_df.reset_index(drop=True)).map(tokenize, batched=True)
    test_ds = Dataset.from_pandas(test_df.reset_index(drop=True)).map(tokenize, batched=True)

    training_args = TrainingArguments(
        output_dir=f"./results_{run_name}",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_steps=50,
    )

    trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=test_ds, compute_metrics=compute_metrics)

    start = time.time()
    trainer.train()
    elapsed_min = (time.time() - start) / 60

    metrics = trainer.evaluate()
    preds = trainer.predict(test_ds)
    pred_labels = np.argmax(preds.predictions, axis=-1)
    cm = confusion_matrix(test_ds["label"], pred_labels, labels=[0, 1])

    print(f"\nTraining time: {elapsed_min:.1f} min")
    print(f"Accuracy:  {metrics['eval_accuracy']:.4f}")
    print(f"Precision: {metrics['eval_precision']:.4f}")
    print(f"Recall:    {metrics['eval_recall']:.4f}")
    print(f"F1 score:  {metrics['eval_f1']:.4f}")
    print(f"Confusion matrix [rows=true, cols=pred, order=FAKE/REAL]:\n{cm}")

    return {
        "run": run_name, "accuracy": metrics["eval_accuracy"], "precision": metrics["eval_precision"],
        "recall": metrics["eval_recall"], "f1": metrics["eval_f1"], "training_time_min": round(elapsed_min, 1),
    }

## Run 1: WITH Reuters (replicates the original 100% result)

In [ ]:
with_reuters_result = train_and_eval("transformer_text", "with_reuters")

## Run 2: WITHOUT Reuters (leakage removed)

In [ ]:
without_reuters_result = train_and_eval("transformer_text_no_reuters", "without_reuters")

## Compare

In [ ]:
comparison = pd.DataFrame([with_reuters_result, without_reuters_result])
comparison["accuracy_drop"] = comparison["accuracy"].iloc[0] - comparison["accuracy"]
comparison.to_csv("reuters_ablation_results.csv", index=False)
print(comparison.to_string(index=False))

drop = with_reuters_result["accuracy"] - without_reuters_result["accuracy"]
print(f"\nAccuracy drop after removing Reuters: {drop*100:.2f} percentage points")
if drop > 0.02:
    print("=> Meaningful drop. Supports the leakage hypothesis: Reuters was inflating the benchmark.")
else:
    print("=> Minimal drop. Reuters alone doesn\'t explain the 100% -- other artifacts likely still present.")

## Download results

In [ ]:
from google.colab import files
files.download("reuters_ablation_results.csv")